In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# import pandas as pd
# data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# data.to_csv("submission.csv", index=False)

**Import modules**

In [ ]:
import re, gc, json, numpy as np, pandas as pd, torch
from difflib import SequenceMatcher
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OUT  = "/kaggle/working/submission.csv"
LABELS = list("ABCDE"); L2I = {l: i for i, l in enumerate(LABELS)}

LLM_NAME      = "Qwen/Qwen3-4B-Instruct-2507"   # non-thinking, ~8GB in fp16
LLM_FALLBACK  = "Qwen/Qwen2.5-3B-Instruct"       # used only if the primary load fails
LLM_BATCH     = 8
LLM_MAXLEN    = 768

DUP_SIM_FLOOR = 0.80    # below this, a retrieval neighbour is ignored (matches your original)
TOP_K         = 30

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "no GPU")


In [ ]:
train_df = pd.read_csv(f"{BASE}/train.csv")
test_df  = pd.read_csv(f"{BASE}/test.csv")
sample   = pd.read_csv(f"{BASE}/sample_submission.csv")

ANSWER_COL = "answer" if "answer" in train_df.columns else train_df.columns[-1]
train_df[ANSWER_COL] = train_df[ANSWER_COL].astype(str).str.strip().str.upper()
train_df = train_df[train_df[ANSWER_COL].isin(LABELS)].reset_index(drop=True)

for c in ["prompt"] + LABELS:
    train_df[c] = train_df[c].fillna("").astype(str)
    test_df[c]  = test_df[c].fillna("").astype(str)

y_train = train_df[ANSWER_COL].map(L2I).values
print("train", train_df.shape, "| test", test_df.shape)

def map_at_3(scores, y):
    t3 = np.argsort(-scores, axis=1)[:, :3]
    return float(sum(np.mean(t3[:, k] == y) / (k + 1) for k in range(3)))

def to_rank(x):
    return np.argsort(np.argsort(x, axis=1), axis=1) / (x.shape[1] - 1)


In [ ]:
def full_text(r):
    return str(r["prompt"]) + " " + " ".join(str(r[c]) for c in LABELS)

tr_txt = [full_text(r) for _, r in train_df.iterrows()]
te_txt = [full_text(r) for _, r in test_df.iterrows()]

wv = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=200_000)
cz = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), sublinear_tf=True, max_features=120_000)
wv.fit(tr_txt + te_txt); cz.fit(tr_txt + te_txt)

Xw_tr, Xc_tr = wv.transform(tr_txt), cz.transform(tr_txt)
Xw_te, Xc_te = wv.transform(te_txt), cz.transform(te_txt)

sim_w_te = cosine_similarity(Xw_te, Xw_tr)
sim_w_tr = cosine_similarity(Xw_tr, Xw_tr); np.fill_diagonal(sim_w_tr, -1.0)   # LOO

def retrieval_scores(sim_matrix):
    n = sim_matrix.shape[0]
    out = np.zeros((n, 5), dtype=np.float64)
    for i in range(n):
        order = np.argsort(-sim_matrix[i])[:TOP_K]
        for j in order:
            s = sim_matrix[i, j]
            if s < DUP_SIM_FLOOR:
                break
            out[i, y_train[j]] += s ** 3
    return out

ret_test_raw = retrieval_scores(sim_w_te)
ret_loo_raw  = retrieval_scores(sim_w_tr)     # leave-one-out proxy on train

ret_loo = np.where(ret_loo_raw.sum(1, keepdims=True) > 0,
                   ret_loo_raw / np.maximum(ret_loo_raw.sum(1, keepdims=True), 1e-9), 0.2)
ret_test = np.where(ret_test_raw.sum(1, keepdims=True) > 0,
                    ret_test_raw / np.maximum(ret_test_raw.sum(1, keepdims=True), 1e-9), 0.2)

print(">>> Retrieval LOO MAP@3 on TRAIN:", round(map_at_3(ret_loo, y_train), 4))
print("    (this is a lower-bound proxy, not your real 0.7581 -- LOO only sees")
print("     duplicates that exist WITHIN train; your real test set apparently")
print("     has many near-twins that live in train, which LOO can't capture.)")


In [ ]:
def load_llm(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"      # required: last position must be the answer slot
    mdl = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
    )
    mdl.eval()
    return tok, mdl

try:
    tok, llm = load_llm(LLM_NAME)
    print("loaded", LLM_NAME)
except Exception as e:
    print("primary load failed:", repr(e), "-> trying fallback")
    tok, llm = load_llm(LLM_FALLBACK)
    print("loaded", LLM_FALLBACK)

letter_ids = []
for L in LABELS:
    ids = set()
    for variant in (L, " " + L):
        t = tok.encode(variant, add_special_tokens=False)
        if len(t) == 1:
            ids.add(t[0])
    if not ids:
        ids = {tok.encode(L, add_special_tokens=False)[0]}
    letter_ids.append(sorted(ids))
print("letter token ids:", letter_ids)
assert all(len(x) > 0 for x in letter_ids), "letter tokenisation produced an empty set"


In [ ]:
def build_prompt(row):
    opts = "\n".join(f"{c}. {row[c]}" for c in LABELS)
    user = ("Answer the multiple choice question. Reply with only the letter "
            f"of the best option.\n\nQuestion: {row['prompt']}\n\nOptions:\n{opts}\n\nAnswer:")
    return tok.apply_chat_template([{"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def llm_score(df):
    prompts = [build_prompt(r) for _, r in df.iterrows()]
    out = np.zeros((len(prompts), 5), dtype=np.float32)
    for s in tqdm(range(0, len(prompts), LLM_BATCH), desc="llm"):
        chunk = prompts[s:s+LLM_BATCH]
        enc = tok(chunk, return_tensors="pt", padding=True,
                 truncation=True, max_length=LLM_MAXLEN).to(llm.device)
        lg = llm(**enc).logits[:, -1, :].float()
        for k, ids in enumerate(letter_ids):
            out[s:s+len(chunk), k] = lg[:, ids].max(dim=-1).values.cpu().numpy()
    out -= out.max(axis=1, keepdims=True)
    p = np.exp(out)
    return p / p.sum(axis=1, keepdims=True)

llm_train = llm_score(train_df)     # zero-shot -> unbiased estimate of test performance
print()
print(">>> LLM zero-shot MAP@3  on TRAIN:", round(map_at_3(llm_train, y_train), 4))
print(">>> LLM zero-shot rank-1 on TRAIN:", round(float(np.mean(llm_train.argmax(1)==y_train)), 4))

llm_test = llm_score(test_df)
np.save("llm_train.npy", llm_train); np.save("llm_test.npy", llm_test)

del llm; gc.collect(); torch.cuda.empty_cache()


In [ ]:
lo, ro = to_rank(llm_train), to_rank(ret_loo)
grid = [(round(w,2), map_at_3(w*ro + (1-w)*lo, y_train)) for w in np.arange(0, 1.01, 0.05)]
best_w, best_s = max(grid, key=lambda t: t[1])

print("w=retrieval weight   MAP@3 (proxy, train)")
for w, s in grid:
    flag = "  <-- best" if w == best_w else ""
    print(f"  {w:.2f}                {s:.4f}{flag}")
print(f"\nbest_w={best_w}  (this proxy MAP@3 is a LOWER bound -- see the note in cell 3)")


In [ ]:
final = best_w * to_rank(ret_test) + (1 - best_w) * to_rank(llm_test)
final = final + np.linspace(1e-9, 0, 5)     # deterministic tie-break

top3 = np.argsort(-final, axis=1)[:, :3]
pred = [" ".join(LABELS[i] for i in r) for r in top3]

id_col, pred_col = sample.columns[0], sample.columns[1]
tid = test_df["id"] if "id" in test_df.columns else test_df[test_df.columns[0]]
sub = pd.DataFrame({id_col: tid.values, pred_col: pred})
sub = sample[[id_col]].merge(sub, on=id_col, how="left")
sub[pred_col] = sub[pred_col].fillna("A B C")
sub.to_csv(OUT, index=False)

assert len(sub) == len(sample), "row count mismatch vs sample_submission"
assert sub[pred_col].str.split().str.len().eq(3).all(), "not every row has exactly 3 letters"
assert sub[pred_col].str.split().map(lambda x: len(set(x))==3).all(), "duplicate letters in a row"

print("rank-1 dist:", dict(Counter(p.split()[0] for p in sub[pred_col])))
print(sub.head(10).to_string(index=False))
print("\nsaved", OUT, "|", len(sub), "rows")
